In [36]:
storage_account = "olistdatalake2027"

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/processed/"

silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/"

StatementMeta(olistsparkjob, 8, 30, Finished, Available, Finished, False)

In [37]:
orders_df = spark.read.parquet(
    bronze_path + "orders/"
)

display(orders_df)

StatementMeta(olistsparkjob, 8, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89f53e7d-c2d2-41b6-be1c-68423a8b1ddc)

In [38]:
print("Total records:", orders_df.count())

print("Total columns:", len(orders_df.columns))

orders_df.printSchema()

StatementMeta(olistsparkjob, 8, 32, Finished, Available, Finished, False)

Total records: 99441
Total columns: 8
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [39]:
from pyspark.sql.functions import col, sum

orders_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in orders_df.columns
]).show()

StatementMeta(olistsparkjob, 8, 33, Finished, Available, Finished, False)

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [40]:
from pyspark.sql.functions import col, to_timestamp

clean_orders = orders_df.dropDuplicates(["order_id"])

clean_orders = clean_orders.filter(
    col("order_id").isNotNull()
)

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for c in date_columns:
    clean_orders = clean_orders.withColumn(
        c,
        to_timestamp(col(c))
    )

display(clean_orders)

StatementMeta(olistsparkjob, 8, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c806983-d347-47ac-a431-07a83c622987)

In [41]:
clean_orders.write \
    .mode("overwrite") \
    .parquet(silver_path + "orders/")

StatementMeta(olistsparkjob, 8, 35, Finished, Available, Finished, False)

In [42]:
silver_orders = spark.read.parquet(
    silver_path + "orders/"
)

print("Silver records:", silver_orders.count())

silver_orders.printSchema()

display(silver_orders.limit(10))

StatementMeta(olistsparkjob, 8, 36, Finished, Available, Finished, False)

Silver records: 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



SynapseWidget(Synapse.DataFrame, 4e9e3e97-0802-4835-82d1-14790ce1d864)

In [43]:

from pyspark.sql import functions as F

# Read customers dataset from Bronze
customers_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("abfss://bronze@olistdatalake2027.dfs.core.windows.net/olist_customers_dataset.csv")

print("Customers Bronze Data Loaded")
print("Total Records:", customers_df.count())

display(customers_df.limit(5))


StatementMeta(olistsparkjob, 8, 37, Finished, Available, Finished, False)

Customers Bronze Data Loaded
Total Records: 99441


SynapseWidget(Synapse.DataFrame, d7f1a51f-b3cb-46a0-af2d-2b6769d67636)

In [44]:

customers_clean = customers_df.dropDuplicates()

# Remove null customer IDs
customers_clean = customers_clean.filter(
    F.col("customer_id").isNotNull()
)

# Remove unwanted spaces
customers_clean = customers_clean.withColumn(
    "customer_city",
    F.trim(F.col("customer_city"))
)

customers_clean = customers_clean.withColumn(
    "customer_state",
    F.trim(F.col("customer_state"))
)

print("Customers cleaning completed")


StatementMeta(olistsparkjob, 8, 38, Finished, Available, Finished, False)

Customers cleaning completed


In [45]:

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

customers_clean.write \
    .mode("overwrite") \
    .parquet(silver_path + "customers/")

print("Customers Silver Layer Saved Successfully!")


StatementMeta(olistsparkjob, 8, 39, Finished, Available, Finished, False)

Customers Silver Layer Saved Successfully!


In [46]:

customers_silver = spark.read.parquet(
    silver_path + "customers/"
)

print("Silver Customers Count:", customers_silver.count())

display(customers_silver.limit(10))


StatementMeta(olistsparkjob, 8, 40, Finished, Available, Finished, False)

Silver Customers Count: 99441


SynapseWidget(Synapse.DataFrame, ee8dab6f-162d-45a6-9dff-6c1b19795d51)

In [47]:

from pyspark.sql import functions as F

bronze_path = "abfss://bronze@olistdatalake2027.dfs.core.windows.net/"

order_items_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_path + "olist_order_items_dataset.csv")

print("Order Items Loaded Successfully")
print("Total Records:", order_items_df.count())

display(order_items_df.limit(5))


StatementMeta(olistsparkjob, 8, 41, Finished, Available, Finished, False)

Order Items Loaded Successfully
Total Records: 112650


SynapseWidget(Synapse.DataFrame, 4cf00d67-10de-4821-bc45-856241e2b7db)

In [48]:

order_items_clean = order_items_df.dropDuplicates()

order_items_clean = order_items_clean.filter(
    F.col("order_id").isNotNull() &
    F.col("product_id").isNotNull()
)

order_items_clean = order_items_clean.filter(
    (F.col("price") >= 0) &
    (F.col("freight_value") >= 0)
)

order_items_clean = order_items_clean.withColumn(
    "shipping_limit_date",
    F.to_timestamp("shipping_limit_date")
)

print("Order Items Cleaning Completed")


StatementMeta(olistsparkjob, 8, 42, Finished, Available, Finished, False)

Order Items Cleaning Completed


In [49]:

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

order_items_clean.write \
    .mode("overwrite") \
    .parquet(silver_path + "order_items/")

print("Order Items Silver Saved Successfully!")


StatementMeta(olistsparkjob, 8, 43, Finished, Available, Finished, False)

Order Items Silver Saved Successfully!


In [50]:

order_items_silver = spark.read.parquet(
    silver_path + "order_items/"
)

print("Total Silver Records:", order_items_silver.count())

display(order_items_silver.limit(10))


StatementMeta(olistsparkjob, 8, 44, Finished, Available, Finished, False)

Total Silver Records: 112650


SynapseWidget(Synapse.DataFrame, 653aa5cd-9b13-467f-a590-ea30b44735e4)

In [51]:

from pyspark.sql import functions as F

bronze_path = "abfss://bronze@olistdatalake2027.dfs.core.windows.net/"

products_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_path + "olist_products_dataset.csv")

print("Products Loaded Successfully")
print("Total Records:", products_df.count())

display(products_df.limit(5))


StatementMeta(olistsparkjob, 8, 45, Finished, Available, Finished, False)

Products Loaded Successfully
Total Records: 32951


SynapseWidget(Synapse.DataFrame, c8baf602-af95-4c56-aef5-81a3eac71627)

In [52]:

products_clean = products_df.dropDuplicates()

products_clean = products_clean.filter(
    F.col("product_id").isNotNull()
)

# Remove spaces
products_clean = products_clean.withColumn(
    "product_category_name",
    F.trim(F.col("product_category_name"))
)

# Convert product dimensions and weight to numeric
numeric_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column_name in numeric_columns:
    products_clean = products_clean.withColumn(
        column_name,
        F.col(column_name).cast("double")
    )

print("Products Cleaning Completed")


StatementMeta(olistsparkjob, 8, 46, Finished, Available, Finished, False)

Products Cleaning Completed


In [53]:

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

products_clean.write \
    .mode("overwrite") \
    .parquet(silver_path + "products/")

print("Products Silver Saved Successfully!")


StatementMeta(olistsparkjob, 8, 47, Finished, Available, Finished, False)

Products Silver Saved Successfully!


In [54]:

products_silver = spark.read.parquet(
    silver_path + "products/"
)

print("Products Silver Count:", products_silver.count())

display(products_silver.limit(10))


StatementMeta(olistsparkjob, 8, 48, Finished, Available, Finished, False)

Products Silver Count: 32951


SynapseWidget(Synapse.DataFrame, 5b8ca633-1bbe-4234-97d7-fe5f600b073c)

In [55]:

from pyspark.sql import functions as F

bronze_path = "abfss://bronze@olistdatalake2027.dfs.core.windows.net/"

payments_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(bronze_path + "olist_order_payments_dataset.csv")

print("Payments Loaded Successfully")
print("Total Records:", payments_df.count())

display(payments_df.limit(5))


StatementMeta(olistsparkjob, 8, 49, Finished, Available, Finished, False)

Payments Loaded Successfully
Total Records: 103886


SynapseWidget(Synapse.DataFrame, 087c7db1-3243-437e-a90d-ceba288ef19a)

In [56]:

payments_clean = payments_df.dropDuplicates()

# Remove invalid order IDs
payments_clean = payments_clean.filter(
    F.col("order_id").isNotNull()
)

# Convert numeric columns
payments_clean = payments_clean.withColumn(
    "payment_sequential",
    F.col("payment_sequential").cast("integer")
).withColumn(
    "payment_installments",
    F.col("payment_installments").cast("integer")
).withColumn(
    "payment_value",
    F.col("payment_value").cast("double")
)

# Remove invalid payment values
payments_clean = payments_clean.filter(
    (F.col("payment_value") >= 0) &
    (F.col("payment_installments") > 0)
)

# Clean payment type
payments_clean = payments_clean.withColumn(
    "payment_type",
    F.lower(F.trim(F.col("payment_type")))
)

print("Payments Cleaning Completed")


StatementMeta(olistsparkjob, 8, 50, Finished, Available, Finished, False)

Payments Cleaning Completed


In [57]:

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

payments_clean.write \
    .mode("overwrite") \
    .parquet(silver_path + "payments/")

print("Payments Silver Saved Successfully!")


StatementMeta(olistsparkjob, 8, 51, Finished, Available, Finished, False)

Payments Silver Saved Successfully!


In [58]:

payments_silver = spark.read.parquet(
    silver_path + "payments/"
)

print("Payments Silver Count:", payments_silver.count())

display(payments_silver.limit(10))


StatementMeta(olistsparkjob, 8, 52, Finished, Available, Finished, False)

Payments Silver Count: 103884


SynapseWidget(Synapse.DataFrame, f4696d17-63e9-4e73-b04e-f5c229061d2e)